Setup Library

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

import joblib
import json

**Tahap 1 — Pengumpulan & Pelabelan Data (Data Gathering & Labeling)**

Load Data

In [2]:
DATA_PATH = "dataset_ct_240_balanced.csv"


df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

Shape: (240, 8)


,student_id,D_score,P_score,A_score,Alg_score,label_ct,total_score,weakest_indicator
0,S001,6,6,5,5,rendah,22,A
1,S002,6,5,6,4,rendah,21,Alg
2,S003,7,6,3,6,rendah,22,A
3,S004,4,5,5,5,rendah,19,D
4,S005,8,6,4,5,rendah,23,A


Cek Struktur Kolom & Missing Value

In [3]:
print(df.columns.tolist())
print("\nMissing value per kolom:")
print(df.isna().sum())

score_cols = ["D_score","P_score","A_score","Alg_score","total_score"]
print("\nDtype score columns:")
print(df[score_cols].dtypes)


['student_id', 'D_score', 'P_score', 'A_score', 'Alg_score', 'label_ct', 'total_score', 'weakest_indicator']

Missing value per kolom:
student_id           0
D_score              0
P_score              0
A_score              0
Alg_score            0
label_ct             0
total_score          0
weakest_indicator    0
dtype: int64

Dtype score columns:
D_score        int64
P_score        int64
A_score        int64
Alg_score      int64
total_score    int64
dtype: object


Validasi Balanced Label

In [4]:
print(df["label_ct"].value_counts())


label_ct
rendah    80
sedang    80
tinggi    80
Name: count, dtype: int64


**Tahap 2 — Pra-pemrosesan Data (Data Preprocessing)**

Tentukan Fitur (X) dan Label (y)

pakai 4 fitur skor pilar CT. Total_score opsional (biasanya tidak dipakai supaya tidak redundan)

In [5]:
X = df[["D_score","P_score","A_score","Alg_score"]].values
y = df["label_ct"].values


Encode Label ke Angka

In [6]:
label_to_int = {"rendah": 0, "sedang": 1, "tinggi": 2}
int_to_label = {0: "rendah", 1: "sedang", 2: "tinggi"}

y_int = np.array([label_to_int[v] for v in y])


Split Data (80% Train, 20% Test)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_int,
    test_size=0.2,      # 20% test -> 48 data dari 240
    random_state=42,    # biar reproducible
    stratify=y_int      # menjaga balance label di train/test
)

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (192, 4) Test: (48, 4)


**Tahap 3 — Pemodelan & Pelatihan (Modeling & Training)**

Latih KNN + Uji beberapa nilai K (Hyperparameter Tuning)

In [8]:
k_candidates = [3, 5, 7, 9, 11]
results = []
models = {}

for k in k_candidates:
    knn = KNeighborsClassifier(n_neighbors=k, metric="euclidean")
    knn.fit(X_train, y_train)

    y_pred = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    results.append({"k": k, "accuracy": acc})
    models[k] = knn

results_df = pd.DataFrame(results).sort_values(by=["accuracy","k"], ascending=[False, True])
results_df


,k,accuracy
4,11,0.833333
1,5,0.812500
0,3,0.791667
2,7,0.791667
3,9,0.791667


Pilih K Terbaik

In [9]:
best_k = int(results_df.iloc[0]["k"])
best_model = models[best_k]

print("Best K:", best_k)


Best K: 11


**Tahap 4 — Evaluasi Model (Model Evaluation)**

Evaluasi Akurasi + Confusion Matrix + Precision/Recall

In [10]:
y_pred_best = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred_best)
cm = confusion_matrix(y_test, y_pred_best)
report = classification_report(
    y_test, y_pred_best,
    target_names=["rendah","sedang","tinggi"],
    digits=4,
    zero_division=0
)

print("Accuracy:", acc)
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", report)


Accuracy: 0.8333333333333334

Confusion Matrix:
 [[13  3  0]
 [ 3 13  0]
 [ 0  2 14]]

Classification Report:
               precision    recall  f1-score   support

      rendah     0.8125    0.8125    0.8125        16
      sedang     0.7222    0.8125    0.7647        16
      tinggi     1.0000    0.8750    0.9333        16

    accuracy                         0.8333        48
   macro avg     0.8449    0.8333    0.8368        48
weighted avg     0.8449    0.8333    0.8368        48



**Tahap 5 — Penyimpanan & Integrasi Model (Deployment)**

Simpan Model ke File .pkl

In [11]:
MODEL_PATH = f"knn_ct_model_k{best_k}.pkl"
joblib.dump(best_model, MODEL_PATH)

# simpan metadata agar gampang integrasi
META_PATH = "knn_ct_meta.json"
meta = {
    "best_k": best_k,
    "features": ["D_score","P_score","A_score","Alg_score"],
    "label_to_int": {"rendah":0,"sedang":1,"tinggi":2},
    "int_to_label": {0:"rendah",1:"sedang",2:"tinggi"},
    "scaling": "none (feature range uniform 0..14)"
}
with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Saved:", MODEL_PATH, META_PATH)


Saved: knn_ct_model_k11.pkl knn_ct_meta.json


Fungsi Predict untuk Dipanggil Web

In [12]:
def predict_overall_level(D_score, P_score, A_score, Alg_score, model_path=MODEL_PATH):
    model = joblib.load(model_path)
    X_new = np.array([[D_score, P_score, A_score, Alg_score]], dtype=int)
    pred_int = int(model.predict(X_new)[0])

    int_to_label = {0:"rendah", 1:"sedang", 2:"tinggi"}
    return int_to_label[pred_int]

# contoh tes
predict_overall_level(13, 1, 2, 12)


'tinggi'